In [ ]:
import os
import json
import cv2
from pathlib import Path
from ultralytics import YOLO

# 请根据你的实际路径修改以下变量
model_path = r"E:\guanxueying\graduation_project\yolo\tobacco\formal\model\m-2560\weights\best.pt"
test_images_dir = r"E:\guanxueying\graduation_project\thesis_fig_table\tobacco\predict"
output_json_dir = r"E:\guanxueying\graduation_project\thesis_fig_table\tobacco\predict"
imgsz = 2560
conf_thres = 0.25
iou_thres = 0.7
class_names = ['leaf', 'root', 'seed']

os.makedirs(output_json_dir, exist_ok=True)
model = YOLO(model_path)

img_files = list(Path(test_images_dir).glob("*.jpg")) + list(Path(test_images_dir).glob("*.png"))
print(f"找到 {len(img_files)} 张图片")

for img_path in img_files:
    results = model(img_path, imgsz=imgsz, conf=conf_thres, iou=iou_thres, verbose=False)
    r = results[0]
    h, w = r.orig_img.shape[:2]
    shapes = []
    if r.masks is not None and hasattr(r.masks, 'xy'):
        polys = r.masks.xy
        if r.boxes is not None:
            cls_ids = r.boxes.cls.cpu().numpy().astype(int)
        else:
            cls_ids = []
        for i, poly in enumerate(polys):
            if len(poly) < 3:
                continue
            pts = [[int(round(p[0])), int(round(p[1]))] for p in poly]
            cls_id = cls_ids[i] if i < len(cls_ids) else 0
            label = class_names[cls_id]
            shapes.append({"label": label, "points": pts, "shape_type": "polygon", "flags": {}})
    json_data = {
        "version": "5.1.0",
        "flags": {},
        "shapes": shapes,
        "imagePath": img_path.name,
        "imageData": None,
        "imageHeight": h,
        "imageWidth": w
    }
    out_json = Path(output_json_dir) / (img_path.stem + ".json")
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(json_data, f, indent=2)
    print(f"已生成: {out_json}")

print("完成！")